# Phishing URL detection model 

This notebook contains all the explorations and ideas I made during the developmnent of a baseline model for phishing detection using the url only.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
FOLDER = Path("data")
PATH = FOLDER / "train.csv"

In [ ]:
df = pd.read_csv(PATH)

## (1) Exploratory data analysis

In [ ]:
import urllib

import matplotlib.pyplot as plt

In [ ]:
print(f"Number of samples: {len(df)}")

Actually, there are some duplicate entries in the dataset. What is more insteresting is that there are several examples with conflicting labels. Unfortunately, I do not have enough time to investigate this issue, so maybe the best thing is to delete all appearances of these urls at this moment.

Actually, there are 50 urls that have 2 different labels in the dataset.

In [ ]:
# There are 50 urls that have 2 different labels

df.groupby("url")["label"].nunique().value_counts()

In [ ]:
# There are actually 8491 urls that have a single label, so they are not duplicates

df.groupby("url")["label"].count().value_counts()

In [ ]:
def collect_urls_with_conflicting_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_counts = df.groupby("url")["label"].nunique()
    return df[df["url"].isin(label_counts[label_counts > 1].index)].sort_values("url")

In [ ]:
conflicting_labels = collect_urls_with_conflicting_labels(df)
conflicting_labels.head(n=10)

In [ ]:
def remove_urls_with_conflicting_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_counts = df.groupby("url")["label"].transform("nunique")
    return df[label_counts == 1]

In [ ]:
df = remove_urls_with_conflicting_labels(df)

In [ ]:
df.head()

In [ ]:
print(f"Number of samples after removing urls with conflicting labels: {len(df)}")

In [ ]:
counts = df["url"].value_counts()
duplicates = counts[counts >= 2]
print(f"urls appear at least twice: {len(duplicates)}")

In [ ]:
duplicates[:5]

In [ ]:
df.drop_duplicates("url", inplace=True, ignore_index=True)

In [ ]:
print(f"Number of samples after removing exact matches: {len(df)}")

In [ ]:
print(f"Number of positive labels: {(df['label'] == 1).sum()}")
print(f"Number of negative labels: {(df['label'] == 0).sum()}")

In [ ]:
def plot_url_length_distribution(ser: pd.Series) -> plt.Figure:
    fig = plt.figure(figsize=(10, 4))
    plt.hist(ser, bins=50, log=True)
    plt.xlabel("URL length")
    plt.ylabel("Count")
    plt.title("Distribution of URL lengths")
    return fig

In [ ]:
fig = plot_url_length_distribution(df["url"].map(len))

In [ ]:
df["url"].apply(len).quantile(0.95)

In [ ]:
df.loc[df["url"].str.len().nlargest(5).index, "url"].tolist()

Maybe it is a good idea to remove query parameters from the url.

In [ ]:
df = df.assign(url_length=df["url"].str.len())

Phishing urls are generally longer, so this is why truncating urls would lead to information loss.

In [ ]:
df.groupby("label")["url_length"].describe()

In [ ]:
df[df["url"].map(lambda x: not x.isascii())]

There are a couple of non-ASCII urls, but not that much (5, actually). All of them are phishing examples. Now the question is what to do with them. I would say, it depends on the model we chose. Actually, as we decode the percent-encoded bytes into Unicode characters, it turns out that there are 18 urls that contain non-ASCII text. For character TF-IDF this can help because the model sees the real character patterns instead of percent-encoding syntax. It is still debateable if we should omit/truncate the outlier urls, but at this point I would still keep them.

* For neural network-based models I would stick to ASCII-characters and replace any other characters with the `<UNK>` symbol (which then will be mapped to an integer later when the urls are converted to input vectors). This way we lose information (many European languages also contain non-ASCII letters), but the vocabulary size remains reasonably small.
* For the baseline model, it depends on the features we chose. My choice here is a logistic regression and I will use the traditional TF-IDF-weighted character n-grams as features. Here I would keep the additional characters.

In [ ]:
def has_query_part(url: str) -> bool:
    parsed = urllib.parse.urlsplit(url)
    return bool(parsed.query)

In [ ]:
# For phishing examples around 25% of the examples has query part, 
# meanwhile, for normal examples this is less than 4%

df["has_query"] = df["url"].map(has_query_part)
df.groupby(["label", "has_query"]).size()

Since the presence of query part in a url can be a good indicator for phishing, I'll keep that part. This way the number of features can be large, but the representation of each example is still sparse so it is not an issue for a regularized logistic regression.

In [ ]:
def transform_url(url: str) -> str:
    url = url.rstrip("/ ")
    try:
        parsed = urllib.parse.urlsplit(url)
        url = parsed.netloc + parsed.path
        if parsed.query:
            url = url + "?" + parsed.query

    except ValueError:
        pass
    
    url = urllib.parse.unquote(url)
    return url.lower()

In [ ]:
df["cleaned_url"] = df["url"].apply(transform_url)

In [ ]:
df.head()

These are the non-ASCII urls after Unicode-decoding of the url strings. There are Greek, Chinese, Turkish and Cyrillic letters, and a three-leaf clover. :)

In [ ]:
df[df["cleaned_url"].map(lambda x: not x.isascii())]

The main reason I would keep all of the characters is that phishing signals often occur below the word level and can be captured at the character level: suspicious substrings, unusual punctuation, misspellings, repeated characters, and the mixture of ASCII/non-ASCII characters can signal the presence of phishing attempts.

It is also clear from the example is that special characters from a specific language that appear in a url that has the domain corresponding to a country where that language is spoken is perfectly normal. Cyrillic letter can appear in the url of a Russian wikipedia article, similarly, a link for translating a Spanish word containing a Spanish character is also expected.

In [ ]:
df["cleaned_url"].apply(len).quantile(0.95)

In [ ]:
df.loc[df["cleaned_url"].str.len().nlargest(5).index, "cleaned_url"].tolist()

There should be further cleaning steps. Due to the time constraints, I leave them as is.

I can also identify near-duplicates by identifying the registered domain.

In [ ]:
df = df.assign(url_length=df["cleaned_url"].str.len())
df.groupby("label")["url_length"].describe()

In [ ]:
df.head()

In [ ]:
# The data cleaning steps will be called independenty, so at this stage we can delete the additional columns

df.drop(columns=["cleaned_url", "url_length", "has_query"], inplace=True)

## Building a baseline model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline

Split the training data into 2 parts:
* train -> use it for model training and selecting the best threshold
* test -> evaluate the final model on this test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[["url"]], 
    df["label"], 
    test_size=0.2,
    random_state=2112,
    shuffle=True,
    stratify=df["label"],
)

In [ ]:
def transform_urls(urls):
    return [transform_url(url) for url in urls]


model = Pipeline([
    (
        "tfidf", 
        TfidfVectorizer(
            analyzer="char", 
            ngram_range=(2, 4), 
            min_df=3,
            max_df=30_000,
            lowercase=False, 
            sublinear_tf=True,
            use_idf=True,
        )),
    ("classifier", LogisticRegression(C=0.5, max_iter=1000, random_state=2112)),
])

The idea here is to 
* train the model by
    * performing data cleaning
    * applying TF-IDF transformation
    * training a logistic regression with L2-regularization using the vectors obtained from TF-IDF weighted (2,3,4)-gram counts
* select the best threshold
    * that optimized a metric (in our case it will be the F1-score)
    * using 5-fold cross-validation, so no separate validation set is needed: we train 5 different models on 80% of the training data and each sample is scored with a model that is not trained on that sample.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2112)

y_train_oof_probs = cross_val_predict(
    model, 
    transform_urls(X_train["url"]), 
    y_train, 
    cv=cv, 
    method="predict_proba"
)[:, 1]

## Select threshold

* Precision: (positive retrieved examples) / (all retrieved examples)
* Recall: (positive retrieved examples) / (all positive examples) = TPR

In [ ]:
import numpy as np
from sklearn.metrics import (
    ConfusionMatrixDisplay, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve, 
    precision_recall_curve,
)

In [ ]:
roc_auc_score(y_train, y_train_oof_probs)

The ideas is to select the threshold that maximizes the F1-score.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_train, y_train_oof_probs)

f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-14)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.5f}")
print(f"CV F1: {f1[best_idx]:.5f}")
print(f"Precision: {precision[best_idx]:.5f}")
print(f"Recall: {recall[best_idx]:.5f}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_train, 
    (y_train_oof_probs >= best_threshold).astype(np.int32),
    cmap=plt.cm.Blues,
    display_labels=[0, 1],
)
plt.show()

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_train, (y_train_oof_probs >= best_threshold).astype(np.int32)).ravel()
fpr = fp / (fp + tn)
print(f"FPR: {fpr}")
print()

Although this looks reasonable, the problem we have is that prevalence in real-life differs significantly from the positive/negative class ratio present in the training data.

Here for the negative examples, 12% of them are classified as phishing url. This means that if we have `100000` real-life examples, `99900` examples would be normal examples, but `~12000` would be flagged as phishing. Precision would be very bad and the high number of false alarms would make the product unusable.

The problem we have here is that the training data is balanced, the positive / negative example ratio is 1:1. However, in real life this ration is more like 1:1000. Using appropriate sample weight, the threshold will increase as false positives become very expensive. This way the model only classifies the very highest-confidence URLs as positive.

In [ ]:
observed_probability = y_train.mean()  # approx 0.5 as the data is balanced
target_probability = 1 / 1001  # this is the prevalence of the positive class in real life

pos_weight = target_probability / observed_probability
neg_weight = (1 - target_probability) / (1 - observed_probability)
sample_weight = np.where(y_train == 1, pos_weight, neg_weight)


precision, recall, thresholds = precision_recall_curve(y_train, y_train_oof_probs, sample_weight=sample_weight)

f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-14)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.5f}")
print(f"CV F1: {f1[best_idx]:.5f}")
print(f"Precision (expected in real-life): {precision[best_idx]:.5f}")
print(f"Recall: {recall[best_idx]:.5f}")

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_train, (y_train_oof_probs >= best_threshold).astype(np.int32)).ravel()
fpr = fp / (fp + tn)
print(f"FPR: {fpr}")
print(f"Precision: {tp / (tp + fp)}")
print(f"Recall: {tp / (tp + fn)}")

In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(
    y_train, 
    (y_train_oof_probs >= best_threshold).astype(np.int32),
    cmap=plt.cm.Blues,
    display_labels=[0, 1],
)
plt.show()

#### Extra part (This part was added later, well beyond the time limit)

The best way of selecting a threshold would be that we fix a desired/tolerated false positive rate, let's say `FPR = 0.001`, then we find the threshold that gives the highest recall while satisfying this constraint. This is more interpretable than the F1-score-based result.

In [ ]:
# precision changes with prevalence
fpr, tpr, thresholds = roc_curve(y_train, y_train_oof_probs)

prevalence = 1 / 1001
precision_real = (tpr * prevalence / (tpr * prevalence + fpr * (1 - prevalence) + 1e-14))
for threshold, recall, fp_rate, precision in zip(thresholds, tpr, fpr, precision_real):
    if fp_rate < 1e-3:
        print(f"{threshold=:.3f} {recall=:.3f} {fp_rate=:.6f} {precision=:.3f}")

So $\tau = 0.837$ is a threshold that would achieve $\text{FPR} < 0.001$ and still maintaining a reasonable recall of `34.1%`.

The threshold that I picked using the weighted F1-score is slightly higher (0.8595) menaing that it is more conservative, resulting in an overly optimistic zero FPR (at least on the training set) with a smaller, `27%` recall, so every 4th phishing url would be detected).

Since I have added these part later, right now I will use the threshold that I obtained by using the weighted F1-score approach.

## Fit final model

In [ ]:
model.fit(transform_urls(X_train["url"]), y_train)

In [ ]:
y_test_prob = model.predict_proba(transform_urls(X_test["url"]))[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(np.int32)

## Evaluate the model

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report, 
    confusion_matrix, 
    confusion_matrix_at_thresholds,
    f1_score,
)

In [ ]:
metrics = classification_report(y_test, y_test_pred, digits=3)
cm = confusion_matrix(y_test, y_test_pred)

In [ ]:
print(metrics)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])

disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
tns, fps, fns, tps, thresholds = confusion_matrix_at_thresholds(y_train, y_train_oof_probs)

# Plot TNs, FPs, FNs and TPs vs Thresholds
plt.figure(figsize=(10, 4))

plt.plot(thresholds, tns, label="True Negatives (TNs)")
plt.plot(thresholds, fps, label="False Positives (FPs)")
plt.plot(thresholds, fns, label="False Negatives (FNs)")
plt.plot(thresholds, tps, label="True Positives (TPs)")
plt.xlabel("Thresholds")
plt.ylabel("Count")
plt.title("Metric versus threshold")
plt.legend()
plt.grid()
plt.show()

In [ ]:
auc = roc_auc_score(y_test, y_test_prob)
print(f"AUC: {auc}")

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_test_prob)

plt.show()

In [ ]:
print("F1 @ 0.5:", f1_score(y_test, y_test_prob >= 0.5))

print(f"F1 @ {best_threshold:.5f}:", f1_score(y_test, y_test_prob >= best_threshold))

## Model export

In [ ]:
import joblib
import os

In [ ]:
os.makedirs("./model", exist_ok=True)
artifact = {
    "model": model,
    "threshold": best_threshold,
}

joblib.dump(artifact, "model/url_phishing_classifier.joblib")

In [ ]:
# Test if the model can be used for inference

import joblib

artifact = joblib.load("model/url_phishing_classifier.joblib")
model = artifact["model"]
threshold = artifact["threshold"]

urls = [
    "http://memory-beta.wikia.com/wiki/Commander",
    "https://www.PayPal-secure.login.tk/webscr?cmd=x",
    "",
    "ww...",
    "htp:/α"
]

probabilities = model.predict_proba(transform_urls(urls))[:, 1]
predictions = (probabilities >= threshold).astype(np.int32)

In [ ]:
probabilities

In [ ]:
predictions

## Model evaluation on the provided unseen test sets

#### Test data

In [ ]:
ddf = pd.read_csv("data/test.csv")

probabilities = model.predict_proba(transform_urls(ddf["url"]))[:, 1]
y_hat = (probabilities >= best_threshold).astype(np.int32)
metrics = classification_report(ddf["label"], y_hat, digits=3)
cm = confusion_matrix(ddf["label"], y_hat)

In [ ]:
print(metrics)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)

plt.show()

In [ ]:
RocCurveDisplay.from_predictions(ddf["label"], probabilities)

plt.show()

The results are very similar to the test part of the training data. Very high precision, `30%` recall.

#### External data

In [ ]:
ddf = pd.read_csv("data/external_test.csv")

probabilities = model.predict_proba(transform_urls(ddf["url"]))[:, 1]
y_hat = (probabilities >= best_threshold).astype(np.int32)
metrics = classification_report(ddf["label"], y_hat, digits=3)
cm = confusion_matrix(ddf["label"], y_hat)

In [ ]:
print(metrics)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)

plt.show()

In [ ]:
RocCurveDisplay.from_predictions(ddf["label"], probabilities)

plt.show()

Here the AUC is down from `0.97` to `0.86`, precision is still high, but recall is down to around `10%` meaning that we only detect every `10th` phishing examples. This is not just a thresholding issue as the AUC is also considerably smaller. The probable reason for this is that the external data comes from a different source, so that the training data does not represent the real distribution of the positive examples and this file contains examples that contain patterns that were not present in the training data. It is not unexpected, we only had less than `4500` positive examples.

#### Extra part (This part was added later, well beyond the time limit)

Before I did a `joblib` export, I thought that an ONNX export is also possible, but unfortunately, the export of the character-based TfIdfVectorizer model is not fully supported yet. There are significant differences between the onnx model scores and the trained Python model scores. Still, it this way we would get a language-agnostic model that can be run from essentially any language.

Another solution would be to use Spark that has Java, Scala and Python APIs, so there is a good chance that a model written and trained in Scala or Python could be exported in a way that is directly tranferable to Java.

In [ ]:
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType

In [ ]:
# Threshold can be added to the export as a metadata field.

options = {
    TfidfVectorizer: {
        "tokenexp": "[^\n]"
    },
    LogisticRegression: {
        "zipmap": False,
    },
}

initial_type = [("input", StringTensorType([None]))]
model_onnx = convert_sklearn(
    model,
    "char_tfidf",
    initial_types=initial_type,
    options=options,
    target_opset=21,
)

metadata = model_onnx.metadata_props.add()
metadata.key = "classification_threshold"
metadata.value = str(best_threshold)

In [ ]:
with open("model/url_phishing_classifier.onnx", "wb") as f:
    f.write(model_onnx.SerializeToString())

In [ ]:
session = rt.InferenceSession("model/url_phishing_classifier.onnx")
metadata = session.get_modelmeta().custom_metadata_map
threshold = float(metadata["classification_threshold"])

input_name = session.get_inputs()[0].name
urls = [
    "http://memory-beta.wikia.com/wiki/Commander",
    "https://www.PayPal-secure.login.tk/webscr?cmd=x",
    "",
    "ww...",
    "htp:/α"
]

pred_onnx = session.run(None, {input_name: transform_urls(urls)})

In [ ]:
probabilities = [float(p) for _, p in pred_onnx[1]]

In [ ]:
probabilities

In [ ]:
[int(p >= threshold) for p in probabilities]